The preprocessing workflow includes:

- Loading the raw CSV using its original CP1252 character encoding
- Standardizing column names using snake_case formatting
- Inspecting and correcting column data types
- Checking for missing values
- Checking for duplicate records
- Validating customer identifier integrity
- Validating product hierarchy integrity
- Investigating `product_id` and `product_name` consistency
- Validating business rules for order dates, sales, quantity, and discounts
- Reviewing product, geographic, customer, and fulfillment categorical features
- Running a final validation gate to verify preprocessing and business logic requirements
- Exporting the cleaned dataset using UTF-8 encoding for downstream analysis in DuckDB

## Load Dataset

Import the dataset into a Pandas DataFrame and perform an initial inspection of its structure before beginning preprocessing and data validation.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/superstore.csv")

df = pd.read_csv(
    data_path,
    encoding="cp1252"
)

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Summary:** The dataset was successfully loaded into Pandas and is ready for preprocessing and validation.

## Standardize column names

Standardize column names using snake_case formatting to improve readability and maintain consistency throughout the project.

In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

**Summary:** All column names were converted to lowercase snake_case format to improve consistency and readability. Standardized names simplify data manipulation and ensure compatibility with SQL queries and downstream analysis.


## Check That Data Types Are Properly Attributed

**Question:** Were numerical, categorical, identifier, and date fields assigned appropriate data types when the dataset was loaded?

Reviewing the inferred data types helps identify columns that Pandas may have interpreted incorrectly. In particular, date columns should use the `datetime64` data type to support temporal calculations, while identifier columns such as `postal_code` should use the `object` data type (or the newer `string` dtype) rather than being treated as numerical measurements.

In [3]:
df.dtypes

row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub_category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object

**Summary:** The initial inspection shows which columns require explicit conversion before further analysis. The date columns and postal code column are corrected in the following sections.

## Convert Postal Codes to Strings

**Question:** Should `postal_code` be treated as a numerical measurement or as a geographic identifier? 

Postal codes are geographic identifiers rather than numerical measurements. They should be stored as strings because they are not used in arithmetic operations and converting them to strings preserves any leading zeros.

In [4]:
df["postal_code"] = df["postal_code"].astype(str)
print(f"Postal Code data type: {df['postal_code'].dtype}")


Postal Code data type: str


**Summary:** The `postal_code` column was converted to a string because postal codes are geographic identifiers rather than numerical measurements. This ensures they are treated as categorical data and prevents unintended numerical operations during downstream analysis.

## Convert Dates to Datetime Data Types

**Question:** Are `order_date` and `ship_date` stored in a format that supports date calculations?

Date columns should be stored as the datetime64 data type to support accurate date arithmetic. This enables calculations such as fulfillment time, extraction of year and month values, chronological comparisons, and other time-based analyses.

In [5]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

print("Updated data types:")
print(df[["order_date", "ship_date"]].dtypes)

Updated data types:
order_date    datetime64[us]
ship_date     datetime64[us]
dtype: object


In [6]:
df[["order_date","ship_date"]].head()

,order_date,ship_date
0,2016-11-08,2016-11-11
1,2016-11-08,2016-11-11
2,2016-06-12,2016-06-16
3,2015-10-11,2015-10-18
4,2015-10-11,2015-10-18


**Summary:** Both `order_date` and `ship_date` were successfully converted to the `datetime64` data type. This enables accurate date arithmetic, such as calculating fulfillment times, extracting years and months, and validating chronological relationships between order and shipping dates.

## Check for Missing Values

**Question:** Are any values missing from the dataset?

Missing values can reduce data quality, introduce bias, and affect downstream analyses. Each column is inspected to determine whether missing records require imputation or removal before analysis.

In [7]:
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

**Summary:** No missing values were identified in any column. Because the dataset is complete, no imputation or row removal was required before proceeding with feature engineering and analysis.

## Check for Duplicated Rows

**Question:** Are any complete transaction rows duplicated?

Fully duplicated rows can inflate sales, quantities, and profits by counting the same transaction multiple times. This check identifies records where every column contains identical values.

In [8]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


**Summary:** No fully duplicated rows were identified. Therefore, no records required removal prior to feature engineering and downstream analysis.

## Business Logic Validation

The following checks evaluate whether the records follow the expected structure and rules of a retail transaction dataset.

### Validate Customer Identifier Integrity

**Question:** Does each `customer_id` consistently correspond to the same `customer_name`?

This check verifies that customer identifiers uniquely identify a single customer and that there are no conflicting customer names associated with the same ID.

In [9]:
customer_mapping = (
    df.groupby("customer_id")["customer_name"]
      .nunique()
)

inconsistent_customer_ids = customer_mapping[
    customer_mapping > 1
]

print(
    f"Customer IDs associated with multiple names: "
    f"{len(inconsistent_customer_ids)}"
)


Customer IDs associated with multiple names: 0


**Summary:** Each `customer_id` maps consistently to a single `customer_name`, supporting the integrity of the customer identifier field.

### Validate Order-Level Structure

**Question:** Can a single `order_id` legitimately contain products from multiple categories and therefore appear multiple times in the dataset?

Retail orders frequently contain multiple line items. This validation confirms that repeated `order_id` values represent legitimate multi-item purchases rather than duplicated transaction records.

In [10]:
(
    df.groupby('order_id')
      .agg(
          num_categories=('category', 'nunique'),
          line_items=('order_id', 'size')
      )
      .query('num_categories > 1')
      .head(5)
)

,num_categories,line_items
order_id,,
CA-2014-100090,2,2
CA-2014-100678,3,4
CA-2014-100706,2,2
CA-2014-100895,2,3
CA-2014-100916,2,3


In [11]:
df[df['order_id'] == 'CA-2014-100678']

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
6568,6569,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,OFF-AR-10001868,Office Supplies,Art,Prang Dustless Chalk Sticks,2.688,2,0.2,1.0080
6569,6570,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,FUR-CH-10002602,Furniture,Chairs,DMI Arturo Collection Mission-style Design Woo...,317.058,3,0.3,-18.1176
6570,6571,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,OFF-EN-10000056,Office Supplies,Envelopes,Cameo Buff Policy Envelopes,149.352,3,0.2,50.4063
6571,6572,CA-2014-100678,2014-04-18,2014-04-22,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Houston,...,77095,Central,TEC-AC-10000474,Technology,Accessories,Kensington Expert Mouse Optical USB Trackball ...,227.976,3,0.2,28.4970


**Summary:** Repeated `order_id` values represent legitimate multi-line purchases rather than duplicated transactions. Individual orders may contain products from multiple categories, so duplicate `order_id` values are expected and should not be treated as duplicate records.

### Validate Product Category Hierarchy

**Question:** Does each `sub_category` consistently belong to a single `category`?

Each product sub-category should map to only one product category.This validation checks for inconsistencies where the same `sub_category` appears under multiple `category` values.

In [12]:
subcategory_mapping = (
    df.groupby("sub_category")["category"]
      .nunique()
)

inconsistent_subcategories = (
    subcategory_mapping[subcategory_mapping > 1]
)

print(
    f"Sub-categories associated with multiple categories: "
    f"{len(inconsistent_subcategories)}"
)

Sub-categories associated with multiple categories: 0


**Summary:** Each `sub_category` maps consistently to a single `category`, confirming the integrity of the product hierarchy.

### Validate Product Identifier Integrity

**Question:** Does each `product_id` consistently correspond to a single `product_name`?

Each product should have a unique identifier that consistently references the same product throughout the dataset. Verifying this relationship helps ensure that product identifiers are reliable and that product-level analyses are based on consistent records.

In [13]:
product_mapping = (
    df.groupby("product_id")["product_name"]
      .nunique()
)

inconsistent_products = product_mapping[product_mapping > 1]

print(
    f"Product IDs associated with multiple product names: "
    f"{len(inconsistent_products)}"
)

Product IDs associated with multiple product names: 32


**Finding:** The initial validation identified 32 `product_id` values associated with multiple `product_name` values. Because this violates the expected one-to-one relationship between product identifiers and product names, the affected records were inspected to better understand the nature of the inconsistencies.


In [14]:
inconsistent_ids = inconsistent_products.index


df[df["product_id"].isin(inconsistent_ids)][
    ["category", "product_id", "product_name"]
].drop_duplicates().sort_values("product_id").head(10)

,category,product_id,product_name
2471,Furniture,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish"
2115,Furniture,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases
66,Furniture,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray"
128,Furniture,FUR-CH-10001146,"Global Task Chair, Black"
1459,Furniture,FUR-FU-10001473,DAX Wood Document Frame
2204,Furniture,FUR-FU-10001473,"Eldon Executive Woodline II Desk Accessories, ..."
3831,Furniture,FUR-FU-10004017,"Executive Impressions 13"" Chairman Wall Clock"
234,Furniture,FUR-FU-10004017,Tenex Contemporary Contur Chairmats for Low an...
1385,Furniture,FUR-FU-10004091,"Eldon 200 Class Desk Accessories, Black"
293,Furniture,FUR-FU-10004091,"Howard Miller 13"" Diameter Goldtone Round Wall..."


**Interpretation:** The representative examples show that several `product_id` values are associated with entirely different `product_name` values rather than minor spelling or formatting variations. This indicates that the inconsistency originates in the source dataset and cannot be resolved reliably without additional business context.

**Summary:** This validation identified inconsistencies between `product_id` and `product_name`. Some `product_id` values were associated with multiple `product_name` values. Without access to the source system or consultation with business stakeholders, the underlying cause of these discrepancies cannot be determined, so they were documented rather than modified during preprocessing. Despite these inconsistencies, the project can confidently proceed because all business analyses are performed at the `category` and `sub_category` levels, whose hierarchical relationship was independently verified to be internally consistent. Consequently, these identifier discrepancies do not affect the validity of the project's feature engineering, descriptive analytics, or predictive modeling workflows.

### Order Date Validation

**Question:** Does every `ship_date` occur on or after its corresponding `order_date`?

Orders should never be shipped before they are placed. This validation confirms that the chronological relationship between `order_date` and `ship_date` is valid for every transaction.

In [15]:
invalid_dates = (df["ship_date"] < df["order_date"]).sum()

print(f"Orders shipped before they were placed: {invalid_dates}")

Orders shipped before they were placed: 0


**Summary:** No transactions were identified where `ship_date` occurred before `order_date`. The chronological relationship between order placement and shipment is valid for all records.

### Validate Positive Sales and Quantity Values

**Question:** Are all `sales` and `quantity` values greater than zero?

Sales and quantity fields should contain positive values because each row represents a completed retail transaction. This check identifies any zero or negative values that may indicate invalid records.

In [16]:
invalid_sales = (df["sales"] <= 0).sum()
invalid_quantity = (df["quantity"] <= 0).sum()

print(f"Sales values less than or equal to zero: {invalid_sales}")                # the zero: o in the output kind of bugs me
print(f"Quantity values less than or equal to zero: {invalid_quantity}")

Sales values less than or equal to zero: 0
Quantity values less than or equal to zero: 0


In [17]:
print(f'There are {(df["sales"] <= 0).sum()} negative sales entries and {(df["quantity"] <= 0).sum()} negative quantity entries.') # I kind of like this line better 

There are 0 negative sales entries and 0 negative quantity entries.


**Summary:** All transactions contain positive `sales` and `quantity` values, indicating that no invalid or zero-value transactions require correction before downstream analysis.

### Check That All Discounts Fall Between 0 and 1

**Question:** Do all `discount` values fall within the valid range of `0` to `1`?

Discount values are stored as proportions, where `0` represents no discount and `1` represents a 100% discount. This validation checks that no values fall outside the expected range.

In [18]:
invalid_discounts = ((df["discount"] < 0) | (df["discount"] > 1)).sum()

print(f"Discount values outside the range 0 to 1: {invalid_discounts}")
print(
    f"Observed discount range: "
    f"{df['discount'].min()} to {df['discount'].max()}"
)

Discount values outside the range 0 to 1: 0
Observed discount range: 0.0 to 0.8


**Summary:** No `discount` values were found outside the valid range of `0` to `1`. The observed discounts range from `0.0` to `0.8`, representing discounts from **0%** to **80%**.

## Categorical Validation

Inspect categorical features to verify that expected categories are present and to understand the structure of the dataset before analysis.

### Product Catalog Validation

**Question:** Are the product-related categorical features consistently labeled and suitable for downstream analysis?

Unique values for `category` and `sub_category` were reviewed to identify spelling errors, inconsistent capitalization, duplicate labels, or unexpected product groupings.

In [19]:
print("Category:")
print(sorted(df['category'].unique()))

print("\nSub-Category:")
print(sorted(df['sub_category'].unique()))


print(f"\nUnique products: {df['product_id'].nunique():,}")

Category:
['Furniture', 'Office Supplies', 'Technology']

Sub-Category:
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']

Unique products: 1,862


**Summary:** Product-related categorical features were successfully inspected. The `category` and `sub_category` variables contain consistent labels with no evidence of spelling errors or inconsistent categorization, providing reliable dimensions for downstream feature engineering and business analysis.

### Geographic Validation

**Question:** What is the geographic coverage of the dataset?

Geographic features were inspected to verify consistent country, region, and state values and to understand the overall geographic coverage of the dataset.


In [20]:
print("Country:")
print(sorted(df['country'].unique()))

print("\nRegion:")
print(sorted(df['region'].unique()))

print("\nState:")
print(sorted(df['state'].unique()))

print(f"\nUnique cities: {df['city'].nunique():,}")
print(f"Unique postal codes: {df['postal_code'].nunique():,}")

Country:
['United States']

Region:
['Central', 'East', 'South', 'West']

State:
['Alabama', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']

Unique cities: 531
Unique postal codes: 631


**Summary:** Geographic features were successfully inspected. The `country`, `region`, and `state` variables contain consistent values, and the dataset spans 531 unique cities and 631 postal codes, providing broad geographic coverage for regional and local sales analyses.

### Customer and Fulfillment Validation

**Question:** Are the customer and fulfillment categories complete and consistently labeled?

Customer-related categorical features were inspected to verify consistent `segment` and `ship_mode` values and to understand the composition of the customer base and available fulfillment methods.

In [21]:
print("Ship Mode:")
print(sorted(df['ship_mode'].unique()))

print("\nSegment:")
print(sorted(df['segment'].unique()))

print(f"\nUnique customer names: {df['customer_name'].nunique():,}")

Ship Mode:
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment:
['Consumer', 'Corporate', 'Home Office']

Unique customer names: 793


**Summary:** Customer-related categorical features were successfully inspected. The `segment` and `ship_mode` variables contain the expected categorical values, and the dataset includes 793 unique customers, providing meaningful dimensions for customer behavior and fulfillment analyses.

### Final Validation Gate

**Question:** Has the dataset satisfied all preprocessing and business logic requirements prior to export?

This final validation gate verifies the critical data quality checks performed throughout the notebook before exporting the cleaned dataset for feature engineering in DuckDB. Documented `product_id`/`product_name` inconsistencies are excluded from this gate because they originate from the source data and do not affect the planned analyses.


In [22]:
# Confirm column names follow lowercase snake_case formatting
assert all(
    column == column.strip()
    and column == column.lower()
    and " " not in column
    and "-" not in column
    for column in df.columns
), "Alert: One or more column names do not follow lowercase snake_case formatting."


# Confirm required columns are present
required_columns = {
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit",
}

missing_columns = required_columns.difference(df.columns)

assert not missing_columns, (
    f"Alert: Required columns are missing: {sorted(missing_columns)}"
)


# Confirm important data types
assert pd.api.types.is_string_dtype(
    df["postal_code"]
), "Alert: postal_code is not stored as a string-compatible data type."

assert pd.api.types.is_datetime64_any_dtype(
    df["order_date"]
), "Alert: order_date is not stored as a datetime."

assert pd.api.types.is_datetime64_any_dtype(
    df["ship_date"]
), "Alert: ship_date is not stored as a datetime."


assert pd.api.types.is_numeric_dtype(
    df["sales"]
), "Alert: sales is not numeric."

assert pd.api.types.is_numeric_dtype(
    df["quantity"]
), "Alert: quantity is not numeric."

assert pd.api.types.is_numeric_dtype(
    df["discount"]
), "Alert: discount is not numeric."

assert pd.api.types.is_numeric_dtype(
    df["profit"]
), "Alert: profit is not numeric."


# Confirm completeness and record integrity
assert not df.empty, "Alert: The dataset contains no records."

assert df.isnull().sum().sum() == 0, (
    "Alert: Missing values were detected."
)

assert df.duplicated().sum() == 0, (
    "Alert: Fully duplicated rows were detected."
)

assert df["row_id"].is_unique, (
    "Alert: row_id contains duplicated values."
)


# Confirm customer identifier integrity
customer_name_counts = (
    df.groupby("customer_id")["customer_name"]
      .nunique()
)

assert customer_name_counts.le(1).all(), (
    "Alert: At least one customer_id is associated with multiple customer names."
)


# Confirm product category hierarchy integrity
subcategory_category_counts = (
    df.groupby("sub_category")["category"]
      .nunique()
)

assert subcategory_category_counts.le(1).all(), (
    "Alert: At least one sub_category is associated with multiple categories."
)


# Confirm chronological order validity
assert df["ship_date"].ge(df["order_date"]).all(), (
    "Alert: At least one ship_date occurs before its order_date."
)


# Confirm transaction value rules
assert df["sales"].gt(0).all(), (
    "Alert: Non-positive sales values were detected."
)

assert df["quantity"].gt(0).all(), (
    "Alert: Non-positive quantity values were detected."
)

assert df["discount"].between(0, 1, inclusive="both").all(), (
    "Alert: Discount values outside the valid range of 0 to 1 were detected."
)


# Confirm expected categorical values
expected_categories = {
    "Furniture",
    "Office Supplies",
    "Technology",
}

expected_segments = {
    "Consumer",
    "Corporate",
    "Home Office",
}

expected_ship_modes = {
    "First Class",
    "Same Day",
    "Second Class",
    "Standard Class",
}

expected_regions = {
    "Central",
    "East",
    "South",
    "West",
}

assert set(df["category"].unique()) == expected_categories, (
    "Alert: Unexpected or missing category values were detected."
)

assert set(df["segment"].unique()) == expected_segments, (
    "Alert: Unexpected or missing segment values were detected."
)

assert set(df["ship_mode"].unique()) == expected_ship_modes, (
    "Alert: Unexpected or missing ship_mode values were detected."
)

assert set(df["region"].unique()) == expected_regions, (
    "Alert: Unexpected or missing region values were detected."
)

assert set(df["country"].unique()) == {"United States"}, (
    "Alert: Unexpected country values were detected."
)


print("Dataset successfully validated and ready for export to DuckDB.")

Dataset successfully validated and ready for export to DuckDB.


**Summary:** The dataset successfully passed all preprocessing and business logic validation checks. Aside from the previously documented source-data inconsistencies between `product_id` and `product_name`, the dataset is clean, internally consistent, and ready for feature engineering and SQL-based analysis in DuckDB.

### Export Clean Dataset

Export the validated dataset as a UTF-8 encoded CSV file for use in DuckDB.

In [23]:
df.to_csv(
    "/Users/danigeiger/projects/e_commerce_duckdb_project/data/superstore_utf8.csv",
    index=False,
    encoding="utf-8"
)

# View cleaned dataset
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Summary:** The cleaned and validated dataset was successfully exported in UTF-8 format and is ready for feature engineering and SQL-based analysis in DuckDB.
